[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/28_moe.ipynb)

# 🔴 Hard: Mixture of Experts (MoE)

Implement a **Mixture of Experts** layer (Mixtral / Switch Transformer style).

### Signature
```python
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, S, D) -> (B, S, D)
```

### Architecture
- `self.router`: `nn.Linear(d_model, num_experts)` — gating network
- `self.experts`: `nn.ModuleList` of MLPs `(Linear→ReLU→Linear)`
- For each token: select top-k experts, compute weighted sum of their outputs

In [1]:
import torch
import torch.nn as nn

In [10]:
# ✏️ YOUR IMPLEMENTATION HERE

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        self.top_k = top_k
        self.router = nn.Linear(d_model,num_experts)
        self.experts = nn.ModuleList(
            [nn.Sequential(
                nn.Linear(d_model,d_ff),
                nn.ReLU(),
                nn.Linear(d_ff,d_model),
            ) for _ in range(num_experts)]
        )

    def forward(self, x):
        logits = self.router(x.flatten(0,-2))
        vals,idx = logits.topk(self.top_k,dim=-1)
        scores = torch.full_like(logits, float("-inf"))
        scores.scatter_(dim=-1,index=idx,src=vals)
        routing_weights = F.softmax(scores,dim=-1)
        final_output = torch.zeros_like(x.flatten(0, -2))
        for i, expert in enumerate(self.experts):
            expert_weights = routing_weights[:, i:i+1]
            
            # Optimization: Only compute for tokens where weight > 0
            if expert_weights.max() > 0:
                expert_out = expert(x.flatten(0, -2)) # (Total_Tokens, d_model)
                final_output += expert_weights * expert_out
        return final_output.view(*x.shape)
        

In [11]:
# 🧪 Debug
moe = MixtureOfExperts(32, 64, num_experts=4, top_k=2)
x = torch.randn(2, 8, 32)
print('Output:', moe(x).shape)
print('Params:', sum(p.numel() for p in moe.parameters()))

NameError: name 'F' is not defined

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('moe')